# 탁구공 탐지 YOLOv8 학습 (Colab, GPU)

런타임 유형을 GPU로 바꾸고(런타임 > 런타임 유형 변경 > T4 GPU) 위에서부터 순서대로 실행하세요.

Roboflow에서 받은 `Ping Pong Ball.v13i.yolov8.zip`을 이 노트북에 업로드해서 진행합니다.

In [ ]:
!pip install -q ultralytics
import torch
print('cuda available:', torch.cuda.is_available())
print('device name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only (런타임 유형을 GPU로 바꾸세요)')

## 1) 데이터셋 업로드

아래 셀 실행하면 파일 선택창이 뜹니다 — `Ping Pong Ball.v13i.yolov8.zip`을 선택하세요.

In [ ]:
from google.colab import files
uploaded = files.upload()  # zip 파일 선택
zip_name = list(uploaded.keys())[0]
print('uploaded:', zip_name)

In [ ]:
import zipfile, os

DATASET_DIR = '/content/ping_pong_ball'
os.makedirs(DATASET_DIR, exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(DATASET_DIR)

!ls {DATASET_DIR}
!cat {DATASET_DIR}/data.yaml

## 2) 학습

`yolov8n.pt`(COCO 사전학습)에서 시작해서 공(Ball) 클래스 하나만 파인튜닝합니다.
GPU면 100 에폭에 보통 10~30분 정도면 끝납니다 (데이터셋 크기 638장 기준).

In [ ]:
!yolo detect train \
    model=yolov8n.pt \
    data={DATASET_DIR}/data.yaml \
    epochs=100 \
    imgsz=640 \
    batch=16 \
    patience=20 \
    project=/content/runs \
    name=ball_yolo

## 3) 결과 확인

학습 곡선/검증 성능(mAP 등)을 그림으로 확인합니다.

In [ ]:
from IPython.display import Image, display

RUN_DIR = '/content/runs/ball_yolo'
display(Image(filename=f'{RUN_DIR}/results.png'))
display(Image(filename=f'{RUN_DIR}/val_batch0_pred.jpg'))

## 4) 가중치 다운로드

`ball_best.pt`로 받아서 `cod/` 폴더에 넣으면 바로 쓸 수 있습니다.

In [ ]:
import shutil
from google.colab import files

shutil.copy(f'{RUN_DIR}/weights/best.pt', '/content/ball_best.pt')
files.download('/content/ball_best.pt')